In [ ]:
import torch
import triton
import triton.language as tl

In [ ]:
def grouped_blocked_matmul_pseudo(A, B, BLOCK_SIZE_M, BLOCK_SIZE_N, GROUP_SIZE):

    M, K = A.shape
    K, N = B.shape
    C = torch.zeros(M, N)

    assert M % BLOCK_SIZE_M == 0, "assuming M divisible by BLOCK_SIZE_M for simplicity"
    assert N % BLOCK_SIZE_N == 0, "assuming N divisible by BLOCK_SIZE_N for simplicity"
    assert (BLOCK_SIZE_M % GROUP_SIZE == 0) and (BLOCK_SIZE_N % GROUP_SIZE == 0)

    # get how many rows and cols we have at the block level
    num_block_rows = M // BLOCK_SIZE_M
    num_block_cols = N // BLOCK_SIZE_N

    num_pids = list(range(num_block_rows * num_block_cols))

    pid_visual = torch.zeros(num_block_rows, num_block_cols)

    for i in num_pids:

        # get number of programs in group
        # this is for all columns, as in our example
        # col_id:     0   1       2   3
        # row_id: 0  [0   2   |   4   6]
        # row_id: 1  [1   3   |   5   7]
        #            -------------------
        # row_id: 2  [8   10  |  12  14]
        # row_id: 3  [9   11  |  13  15]
        num_pid_in_group = GROUP_SIZE * num_block_cols

        # which group are we in?
        group_id = i // num_pid_in_group

        # starting row in m dimension for this specific group
        first_pid_m = group_id * GROUP_SIZE

        # what is the true groupsize incase not divisible
        group_size_m = min(num_block_rows - first_pid_m, GROUP_SIZE)

        where_in_group = i % num_pid_in_group

        which_row = where_in_group % group_size_m

        # this is the row within the group, and we shift it over to the actual index of our block matrix
        pid_m = first_pid_m + which_row

        # now we do the same thing for the column that this pid is supposed to compute
        # we can just do first i % num_pid_in_group (same as before this is redundant)
        # if i == 0, i % 8 = 0
        # if i == 1, i % 8 = 1
        # if i == 2, i % 8 = 2
        # ...
        # if i == 7, i % 8 = 7
        # if i == 8, i % 8 = 0
        where_in_group = i % num_pid_in_group

        # and just do division to get our actual row we are in!
        # where_in_group // group_size_m
        # if i == 0, where_in_group = i % 8 = 0 // 2 = 0
        # if i == 1, where_in_group = i % 8 = 1 // 2 = 0
        # if i == 2, where_in_group = i % 8 = 2 // 2 = 1
        # if i == 3, where_in_group = i % 8 = 3 // 2 = 1
        # if i == 4, where_in_group = i % 8 = 4 // 2 = 2
        # if i == 5, where_in_group = i % 8 = 5 // 2 = 2
        # ...
        pid_n = where_in_group // group_size_m


        # pid 0: (0, 0)
        # pid 1: (1, 0)
        # pid 2: (0, 1)
        # pid 3: (1, 1)
        # pid 4: (0, 2)
        # pid 5: (1, 2)
        # pid 6: (0, 3)
        # pid 7: (1, 3)
        # pid 8: (2, 0)
        # pid 9: (3, 0)
        # pid 10: (2, 1)
        # pid 11: (3, 1)
        # pid 12: (2, 2)
        # pid 13: (3, 2)
        # pid 14: (2, 3)
        # pid 15: (3, 3)

        pid_visual[pid_m, pid_n] = i

        m = pid_m * BLOCK_SIZE_M
        n = pid_n * BLOCK_SIZE_N

        block_A = A[m : m + BLOCK_SIZE_M, :]
        block_B = B[:, n : n + BLOCK_SIZE_N]

        C[m : m + BLOCK_SIZE_M, n : n + BLOCK_SIZE_N] = block_A @ block_B

    return C, pid_visual

triton kernel

In [ ]:
def get_cuda_autotune_config():
    return [
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 64, 'GROUP_SIZE_M': 8}, num_stages=3, num_warps=8),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 256, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 128, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 128, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=4, num_warps=4),
        triton.Config({'BLOCK_SIZE_M': 64, 'BLOCK_SIZE_N': 32, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=5, num_warps=2),
        triton.Config({'BLOCK_SIZE_M': 32, 'BLOCK_SIZE_N': 64, 'BLOCK_SIZE_K': 32, 'GROUP_SIZE_M': 8}, num_stages=5, num_warps=2),
    ]
@triton.autotune(configs = get_cuda_autotune_config(), key=['M', 'N', 'K'])

In [ ]:
@triton.jit
def grouped_blocked_matmul_kernel(
        A_ptr, B_ptr, C_ptr,
        M,N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        BLOCK_SIZE_M: tl.constexpr,
        BLOCK_SIZE_N: tl.constexpr,
        BLOCK_SIZE_K: tl.constexpr,
        GROUP_SIZE_M: tl.constexpr,
):
    pid = tl.program_id(0)

    num_pid_m = tl.cdiv(M, BLOCK_SIZE_M)
    num_pid_n = tl.cdiv(N, BLOCK_SIZE_N)
    num_pid_in_group = GROUP_SIZE_M * num_pid_n

    group_id = pid // num_pid_in_group

    first_pid_m = group_id * GROUP_SIZE_M
    group_size_m = min(num_pid_m - first_pid_m, GROUP_SIZE_M)
    pid_m = first_pid_m + ((pid % num_pid_in_group) % group_size_m)
    pid_n = (pid % num_pid_in_group) // group_size_m

    offs_m = pid_m * BLOCK_SIZE_M + tl.arange(0, BLOCK_SIZE_M)
    offs_n = pid_n * BLOCK_SIZE_N + tl.arange(0, BLOCK_SIZE_N)
    offs_k = tl.arange(0, BLOCK_SIZE_K)

    a_ptrs = A_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = B_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    accumulator = tl.zeros((BLOCK_SIZE_M, BLOCK_SIZE_N), dtype=tl.float32)

    for k in range(0, tl.cdiv(K, BLOCK_SIZE_K)):
        a_mask = (offs_m[:, None] < M) & (offs_k[None, :] < K - k * BLOCK_SIZE_K)
        a = tl.load(a_ptrs, mask=a_mask, other=0.0)

        b_mask = (offs_k[:, None] < K - k * BLOCK_SIZE_K) & (offs_n[None, :] < N)
        b = tl.load(b_ptrs, mask=b_mask, other = 0.0)

        accumulator += tl.dot(a, b, allow_tf32=False)   # vlock_size_m x block_size_n

        a_ptrs += BLOCK_SIZE_K * stride_ak
        b_ptrs += BLOCK_SIZE_K * stride_bk

    c = accumulator.to(A_ptr.dtype.element_ty)

    c_offsets = stride_cm * offs_m[:, None] + stride_cn * offs_n[None, :]

    c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    tl.store(C_ptr + c_offsets, c, mask=c_mask)


In [ ]:
def fused_grouped_matmul(a, b):

    # get dimensions
    M, K = a.shape
    K, N = b.shape

    assert (a.dtype == torch.float32) or (a.dtype == torch.float16),    "only support float32 or float16 ops"
    assert a.dtype == b.dtype, f'a and b have different precision types {a.dtype} and {b.dtype}'

    # initialize output tensor with same dtype as input
    c = torch.empty((M, N), device=a.device, dtype=a.dtype)

    # define grid
    grid = lambda meta: (triton.cdiv(M, meta['BLOCK_SIZE_M']) * triton.cdiv(N, meta['BLOCK_SIZE_N']), )

    # launch kernel
    grouped_blocked_matmul_kernel[grid](
        a, b, c,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        DTYPE_FLAG=0 if a.dtype == torch.float32 else 1
    )

    return c

In [ ]:
# test
@pytest.mark.parametrize("M, N, K", [
    (64, 64, 64),
    (128, 128, 128),
    (256, 256, 256),
    (512, 512, 512),
    (1024, 1024, 1024),
    (256, 512, 128),
    (512, 256, 128)
])

def test_matmul(M, N, K):
    # create random inputs
    a = torch.randn(size=(M, K), dtype=torch.float16, device="cuda")
    b = torch.randn(size=(K, N), dtype=torch.float16, device="cuda")

    out_ref = torch.matmul(a, b)

    # your fused kernel
    out = fused_grouped_matmul(a, b)

    # validate results
    torch.allclose(out, out_ref, atol=1e-2, rtol=1e-2)

    # log useful info
    max_diff = float(torch.max(torch.abs(out - out_ref)))
    print(f'matmul success: M={M}, N={N}, K={K} | max diff = {max_diff:.4f}')

In [ ]:
configs = [
    triton.testing.Benchmark(
        x_names = ["M", "N", "K"],
        x_vals = [512 * i for i in range(2, 16)],
        line_arg = "provider",
        line_vals = ["torch", "blocked_matmul", "grouped_matmul"],
        line_names = ["PyTorch", "Matmul No Groups", "Matmul Grouped"],
        styles = [("blues", "-"), ("purple", "-"), ("green", "-")],
        ylabel = "TFLOPS",
        plot_name = "matmul",
        args = {},
    )
]

In [ ]:
@triton.testing.perf_report(configs)
def benchmark_matmul(M, N, K, provider):

    if provider == 'torch':
        a = torch.randn((M, K), device="cuda", dtype=torch.float16)
        b = torch.randn((K, N), device="cuda", dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: torch.matmul(a, b))

    if provider == 'blocked_matmul':
        a = torch.randn((M, K), device="cuda", dtype=torch.float16)
        b = torch.randn((K, N), device="cuda", dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: blocked_matmul(a, b))

    if provider == "grouped_matmul":
        a = torch.randn((M, K), device="cuda", dtype=torch.float16)
        b = torch.randn((K, N), device="cuda", dtype=torch.float16)
        ms = triton.testing.do_bench(lambda: fused_grouped_matmul(a, b))

    perf = lambda ms: 2 * M * N * K * 1e-12 / (ms * 1e-3)
    retrun perm(ms)

In [ ]:
if __name__ == "__main__":

    print("running test...")
    test_path = Path(__file__).resolve()
    pytest.main([f'{test_path}::test_matmul', "-v", "-s"])

    print("running benchmark...")
    benchmark_matmul.run(show_plots=True)